In [ ]:

import math
import random
from copy import deepcopy

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed: int, deterministic=False):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

DEVICE = get_device()
print("Usando dispositivo:", DEVICE)
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória GPU disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Rodando na CPU")


Usando dispositivo: cuda
GPU: NVIDIA GeForce GTX 1650
Memória GPU disponível: 4.1 GB


In [ ]:
DATA_ROOT = "./data"

FAST_DEV_RUN = False

if FAST_DEV_RUN:
    LABEL_BUDGETS = (25,)
    NUM_EPOCHS = 20
    MU = 3
else:
    LABEL_BUDGETS = (1, 4, 25, 400)
    NUM_EPOCHS = 70
    MU = 7

BATCH_SIZE = 64
SEED = 123

ENABLE_VISUALIZATION = True
ENABLE_PSEUDO_STATS = False


In [ ]:

class Cutout(object):
    def __init__(self, mask_size, p=1.0):
        self.mask_size = mask_size
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        if not isinstance(img, torch.Tensor):
            img = transforms.functional.to_tensor(img)
        c, h, w = img.shape
        mask_size_half = self.mask_size // 2
        cx = random.randint(0, w - 1)
        cy = random.randint(0, h - 1)
        x1 = max(0, cx - mask_size_half)
        y1 = max(0, cy - mask_size_half)
        x2 = min(w, cx + mask_size_half)
        y2 = min(h, cy + mask_size_half)
        img[:, y1:y2, x1:x2] = 0.0
        return img


class CustomRandAugment:
    def __init__(self, n=2, m=10):
        self.n = n  # número de transformações
        self.m = m  # magnitude (0-10)


        self.transforms = [
            ('autocontrast', lambda img, mag: transforms.functional.autocontrast(img)),
            ('brightness', lambda img, mag: transforms.functional.adjust_brightness(img, 0.05 + (0.95-0.05) * mag/10)),
            ('color', lambda img, mag: transforms.functional.adjust_saturation(img, 0.05 + (0.95-0.05) * mag/10)),
            ('contrast', lambda img, mag: transforms.functional.adjust_contrast(img, 0.05 + (0.95-0.05) * mag/10)),
            ('equalize', lambda img, mag: transforms.functional.equalize(img)),
            ('identity', lambda img, mag: img),
            ('posterize', lambda img, mag: transforms.functional.posterize(img, int(4 + (8-4) * mag/10))),
            ('rotate', lambda img, mag: transforms.functional.rotate(img, -30 + 60 * mag/10)),
            ('sharpness', lambda img, mag: transforms.functional.adjust_sharpness(img, 0.05 + (0.95-0.05) * mag/10)),
            ('shear_x', lambda img, mag: transforms.functional.affine(img, angle=0, translate=[0,0], scale=1, shear=[-30 + 60 * mag/10, 0])),
            ('shear_y', lambda img, mag: transforms.functional.affine(img, angle=0, translate=[0,0], scale=1, shear=[0, -30 + 60 * mag/10])),
            ('solarize', lambda img, mag: transforms.functional.solarize(img, int(255 * mag/10))),
            ('translate_x', lambda img, mag: transforms.functional.affine(img, angle=0, translate=[int(img.size[0] * (-0.3 + 0.6 * mag/10)), 0], scale=1, shear=[0,0])),
            ('translate_y', lambda img, mag: transforms.functional.affine(img, angle=0, translate=[0, int(img.size[1] * (-0.3 + 0.6 * mag/10))], scale=1, shear=[0,0])),
        ]

    def __call__(self, img):
        # Seleciona aleatoriamente n transformações das 14 disponíveis
        selected_ops = random.sample(self.transforms, self.n)

        for name, op in selected_ops:
            # 50% de chance de aplicar cada transformação selecionada
            if random.random() < 0.5:
                try:
                    img = op(img, self.m)
                except:
                    pass
        return img


def get_transforms():
    mean = torch.tensor([0.4914, 0.4822, 0.4465])
    std  = torch.tensor([0.2023, 0.1994, 0.2010])
    print(f"CIFAR-10 mean: {mean.tolist()}, std: {std.tolist()}")

    # Weak augmentation: flip horizontal 50% + translate até 12.5%
    weak_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=0, translate=(0.125, 0.125)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    # Strong augmentation: RandAugment (14 transformações) + Cutout
    strong_transform = transforms.Compose([
        CustomRandAugment(n=2, m=10),
        transforms.ToTensor(),
        Cutout(mask_size=16, p=1.0),
        transforms.Normalize(mean, std),
    ])

    return weak_transform, strong_transform, mean, std


def split_labeled_unlabeled(dataset, labels_per_class, num_classes=10):
    labels = torch.tensor(dataset.targets)
    labeled_idx = []
    for c in range(num_classes):
        idx_c = (labels == c).nonzero(as_tuple=False).view(-1)
        idx_c = idx_c[torch.randperm(len(idx_c))]
        take = min(labels_per_class, len(idx_c))
        labeled_idx.extend(idx_c[:take].tolist())
    labeled_idx = sorted(labeled_idx)
    unlabeled_idx = list(range(len(dataset)))
    return labeled_idx, unlabeled_idx


class LabeledCIFAR10(Dataset):
    def __init__(self, root, idxs, transform=None, download=False):
        self.dataset = torchvision.datasets.CIFAR10(
            root=root, train=True, download=download)
        self.idxs = idxs
        self.transform = transform

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, i):
        idx = self.idxs[i]
        img, target = self.dataset[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, target


class UnlabeledCIFAR10(Dataset):
    def __init__(self, root, idxs, weak_transform=None, strong_transform=None, download=False):
        self.dataset = torchvision.datasets.CIFAR10(
            root=root, train=True, download=download)
        self.idxs = idxs
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, i):
        idx = self.idxs[i]
        img, _ = self.dataset[idx]
        img_w = self.weak_transform(img) if self.weak_transform else transforms.ToTensor()(img)
        img_s = self.strong_transform(img) if self.strong_transform else img_w
        return img_w, img_s


In [4]:
def build_resnet18_cifar10():
    model = torchvision.models.resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()
    return model


In [ ]:
# %%
class SupervisedTrainer:
    def __init__(
        self,
        model,
        labeled_loader,
        test_loader,
        device,
        lr=0.03,
        weight_decay=5e-4,
        num_epochs=1024,
    ):
        self.model = model.to(device)
        self.device = device
        self.labeled_loader = labeled_loader
        self.test_loader = test_loader

        # SGD com momentum=0.9, lr=0.03, weight_decay=0.0005, sem Nesterov
        self.optimizer = torch.optim.SGD(
            self.model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay,
            nesterov=False,
        )

        # Cosine decay: η·cos(7πk/16K)
        K = num_epochs * len(labeled_loader)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lr_lambda=lambda step: math.cos(7 * math.pi * step / (16 * K))
        )
        self.history = {"epoch": [], "loss": [], "acc": []}

    def train(self, num_epochs, log_every=5):
        for epoch in range(num_epochs):
            self.model.train()
            epoch_loss = 0.0
            num_batches = 0

            for x, y in self.labeled_loader:
                x = x.to(self.device)
                y = y.to(self.device)

                logits = self.model(x)
                loss = F.cross_entropy(logits, y)

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                epoch_loss += loss.item()
                num_batches += 1

            epoch_loss /= max(1, num_batches)
            acc = self.evaluate()

            self.history["epoch"].append(epoch + 1)
            self.history["loss"].append(epoch_loss)
            self.history["acc"].append(acc)

            if (epoch + 1) % log_every == 0 or epoch == 0 or epoch + 1 == num_epochs:
                print(
                    f"[Supervised] Epoch {epoch+1}/{num_epochs} "
                    f"- Loss: {epoch_loss:.4f} - Acc: {acc:.2f}%"
                )

        return self.model

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        correct = 0
        total = 0
        for x, y in self.test_loader:
            x = x.to(self.device)
            y = y.to(self.device)
            logits = self.model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
        return 100.0 * correct / total


In [ ]:
class FixMatchTrainer:
    """
    Implementa o laço de treinamento do FixMatch:

    - Loss supervisionada em exemplos rotulados fracos.
    - Pseudo-rótulos gerados em weak augment dos não rotulados.
    - Consistência: forçar strong augment a seguir o pseudo-rótulo
      quando confiança >= tau.
    """
    def __init__(
        self,
        model,
        labeled_loader,
        unlabeled_loader,
        test_loader,
        device,
        lambda_u=0.5,
        tau=0.95,
        num_epochs=1024,
        total_steps=None,
        mu=7,
        lr=0.03,
        weight_decay=5e-4,
    ):
        self.model = model.to(device)
        self.device = device
        self.labeled_loader = labeled_loader
        self.unlabeled_loader = unlabeled_loader
        self.test_loader = test_loader
        self.lambda_u = lambda_u
        self.tau = tau
        self.mu = mu
        self.num_classes = 10
        self.da_momentum = 0.999
        self.p_model = torch.ones(self.num_classes, device=self.device) / self.num_classes
        self.p_target = torch.ones(self.num_classes, device=self.device) / self.num_classes

        # SGD com momentum=0.9, lr=0.03, weight_decay=0.0005, sem Nesterov
        self.optimizer = torch.optim.SGD(
            self.model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay,
            nesterov=False,
        )

        # Cosine decay: η·cos(7πk/16K)
        K = num_epochs * len(unlabeled_loader)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lr_lambda=lambda step: math.cos(7 * math.pi * step / (16 * K))
        )

        self.history = {
            "epoch": [],
            "sup_loss": [],
            "unsup_loss": [],
            "total_loss": [],
            "acc": [],
            "pseudo_label_coverage": [],
        }



    def train(self, num_epochs):
      self.model.train()
      global_step = 0
      labeled_iter = iter(self.labeled_loader)
      self.optimizer.zero_grad()

      for epoch in range(num_epochs):
          current_tau = self.tau

          sup_loss_epoch = 0.0
          unsup_loss_epoch = 0.0
          total_loss_epoch = 0.0
          pseudo_cov_epoch = 0.0
          num_batches = 0

          for (uw, us) in self.unlabeled_loader:
              # Garante dados rotulados contínuos
              try:
                  x_l, y_l = next(labeled_iter)
              except StopIteration:
                  labeled_iter = iter(self.labeled_loader)
                  x_l, y_l = next(labeled_iter)

              x_l = x_l.to(self.device); y_l = y_l.to(self.device)
              uw  = uw.to(self.device);  us  = us.to(self.device)

              # Pseudo-labels: weak augment → confiança
              self.model.eval()
              with torch.no_grad():
                  probs_w = F.softmax(self.model(uw), dim=1)

                  # Distribution Alignment
                  self.p_model = self.p_model * self.da_momentum + (1 - self.da_momentum) * probs_w.mean(dim=0)
                  adjust = (self.p_target / (self.p_model + 1e-6))
                  probs_w = probs_w * adjust
                  probs_w = probs_w / probs_w.sum(dim=1, keepdim=True)

                  max_probs, pseudo_labels = probs_w.max(dim=1)
                  mask = (max_probs >= current_tau).float()

              # MixMatch: concatena labeled + unlabeled strong
              inputs = torch.cat([x_l, us], dim=0)
              logits_all = self.model(inputs)
              logits_l  = logits_all[:x_l.size(0)]
              logits_us = logits_all[x_l.size(0):]

              # Loss supervisionada
              loss_sup = F.cross_entropy(logits_l, y_l)

              # Loss não-supervisionada (apenas amostras confiantes)
              loss_unsup_all = F.cross_entropy(logits_us, pseudo_labels, reduction='none')
              if mask.sum() > 0:
                  loss_unsup = (loss_unsup_all * mask).mean()
              else:
                  loss_unsup = torch.tensor(0.0, device=self.device)

              loss = loss_sup + self.lambda_u * loss_unsup

              loss.backward()
              torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
              self.optimizer.step()
              self.scheduler.step()
              self.optimizer.zero_grad()

              global_step += 1

              sup_loss_epoch += loss_sup.item()
              unsup_loss_epoch += loss_unsup.item()
              total_loss_epoch += loss.item()
              pseudo_cov_epoch += mask.mean().item()
              num_batches += 1

          sup_loss_epoch /= max(1, num_batches)
          unsup_loss_epoch /= max(1, num_batches)
          total_loss_epoch /= max(1, num_batches)
          pseudo_cov_epoch /= max(1, num_batches)

          acc = self.evaluate()
          print(
              f"Epoch {epoch+1}/{num_epochs} "
              f"- L_sup: {sup_loss_epoch:.4f} "
              f"- L_unsup: {unsup_loss_epoch:.4f} "
              f"- L_total: {total_loss_epoch:.4f} "
              f"- Mask_cov: {pseudo_cov_epoch:.3f} "
              f"- Acc: {acc:.2f}%"
          )

          self.history["epoch"].append(epoch + 1)
          self.history["sup_loss"].append(sup_loss_epoch)
          self.history["unsup_loss"].append(unsup_loss_epoch)
          self.history["total_loss"].append(total_loss_epoch)
          self.history["acc"].append(acc)
          self.history["pseudo_label_coverage"].append(pseudo_cov_epoch)

      return self.model


    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        correct = 0
        total = 0
        for x, y in self.test_loader:
            x = x.to(self.device)
            y = y.to(self.device)
            logits = self.model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
        return 100.0 * correct / total

    @torch.no_grad()
    def pseudo_label_stats(self):
      self.model.eval()
      counts = torch.zeros(10, device=self.device)
      total = 0
      for uw, us in self.unlabeled_loader:
          uw = uw.to(self.device)
          probs = F.softmax(self.model(uw), dim=1)
          _, pseudo = probs.max(dim=1)
          for c in range(10):
              counts[c] += (pseudo == c).sum()
          total += uw.size(0)
          if total > 50000:
              break
      dist = (counts / counts.sum()).cpu().numpy()
      print("Distribuição de pseudo-rótulos:", dist)


In [ ]:
def plot_supervised_history(history, title="Supervised"):
  epochs = history["epoch"]

  plt.figure()
  plt.plot(epochs, history["loss"], label="Train Loss")
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.title(f"{title} - Loss")
  plt.legend()
  plt.show()

  plt.figure()
  plt.plot(epochs, history["acc"], label="Test Acc")
  plt.xlabel("Epoch")
  plt.ylabel("Accuracy (%)")
  plt.title(f"{title} - Accuracy")
  plt.legend()
  plt.show()


def plot_fixmatch_history(history, title="FixMatch"):
  epochs = history["epoch"]

  plt.figure()
  plt.plot(epochs, history["sup_loss"], label="Sup Loss")
  plt.plot(epochs, history["unsup_loss"], label="Unsup Loss")
  plt.plot(epochs, history["total_loss"], label="Total Loss")
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.title(f"{title} - Losses")
  plt.legend()
  plt.show()

  plt.figure()
  plt.plot(epochs, history["acc"], label="Accuracy")
  plt.xlabel("Epoch")
  plt.ylabel("Accuracy (%)")
  plt.title(f"{title} - Accuracy")
  plt.legend()
  plt.show()

  plt.figure()
  plt.plot(epochs, history["pseudo_label_coverage"], label="Pseudo-label coverage")
  plt.xlabel("Epoch")
  plt.ylabel("Frac conf >= tau")
  plt.title(f"{title} - Pseudo-label usage")
  plt.legend()
  plt.show()

def visualize_pseudo_labels_fixmatch(
    fixmatch_trainer,
    mean,
    std,
    unlabeled_loader,
    class_names=None,
    max_images=16,
    tau=None,
    title="Pseudo-rótulos em dados não rotulados"
):
    """Visualiza pseudo-labels: weak augment vs strong augment predictions"""
    if class_names is None:
        class_names = [str(i) for i in range(10)]

    device = fixmatch_trainer.device
    model = fixmatch_trainer.model.to(device).eval()

    if tau is None:
        tau = fixmatch_trainer.tau

    uw, us = next(iter(unlabeled_loader))
    uw = uw.to(device); us = us.to(device)

    with torch.no_grad():
        # Pseudo-labels do weak augment
        probs_w = F.softmax(model(uw), dim=1)
        # Aplica Distribution Alignment aprendido
        adjust = (fixmatch_trainer.p_target.to(device) /
                  (fixmatch_trainer.p_model.to(device) + 1e-6))
        probs_w = probs_w * adjust
        probs_w = probs_w / probs_w.sum(dim=1, keepdim=True)

        max_probs, pseudo_labels = probs_w.max(dim=1)
        # Predições do strong augment (para comparar consistência)
        strong_preds = model(us).argmax(dim=1)

    mask = max_probs >= tau
    idxs = torch.nonzero(mask).view(-1)
    if len(idxs) == 0:
        print("Nenhuma amostra com confiança >= tau nesse batch.")
        return

    idxs = idxs[:max_images]
    uw = uw[idxs].cpu()
    pseudo_labels = pseudo_labels[idxs].cpu()
    strong_preds = strong_preds[idxs].cpu()
    max_probs = max_probs[idxs].cpu()

    mean_t = (mean if isinstance(mean, torch.Tensor) else torch.tensor(mean)).view(3,1,1)
    std_t  = (std  if isinstance(std,  torch.Tensor) else torch.tensor(std)).view(3,1,1)

    n = len(idxs)
    cols = min(8, n); rows = (n + cols - 1) // cols
    plt.figure(figsize=(2.5 * cols, 2.8 * rows))
    for i in range(n):
        img = torch.clamp(uw[i] * std_t + mean_t, 0.0, 1.0)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img.permute(1, 2, 0).numpy())
        pl = class_names[pseudo_labels[i].item()]
        sp = class_names[strong_preds[i].item()]
        conf = max_probs[i].item()
        ok = "✓" if pl == sp else "✗"
        plt.title(f"PL: {pl}\nStrong: {sp} {ok}\nconf={conf:.2f}", fontsize=8)
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()



def plot_overall_accuracy(results):
    """Compara acurácia final: Supervised vs FixMatch por quantidade de rótulos"""
    label_counts = sorted(results.keys())
    sup_accs = [results[k]["supervised_acc"] for k in label_counts]
    fm_accs = [results[k]["fixmatch_acc"] for k in label_counts]

    plt.figure()
    plt.plot(label_counts, sup_accs, marker="o", label="Supervised")
    plt.plot(label_counts, fm_accs, marker="o", label="FixMatch")
    plt.xscale("log")
    plt.xticks(label_counts, label_counts)
    plt.xlabel("# rótulos por classe (log)")
    plt.ylabel("Acurácia de teste (%)")
    plt.title("Supervised vs FixMatch em diferentes quantidades de rótulos")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

    print("Resumo por # rótulos/classe:")
    for k in label_counts:
        print(
            f"{k:4d} | Sup: {results[k]['supervised_acc']:.2f}% "
            f"| FixMatch: {results[k]['fixmatch_acc']:.2f}%"
        )

def visualize_labeled_predictions(
    model,
    device,
    data_loader,
    mean,
    std,
    class_names=None,
    max_images=16,
    title="Predições em dados rotulados"
):
    """Mostra predições vs ground truth em dados rotulados"""
    if class_names is None:
        class_names = [str(i) for i in range(10)]

    model.eval()
    mean_t = mean.view(3,1,1) if isinstance(mean, torch.Tensor) else torch.tensor(mean).view(3,1,1)
    std_t  = std.view(3,1,1)  if isinstance(std,  torch.Tensor) else torch.tensor(std).view(3,1,1)


    # Pega um batch para visualizar
    x, y = next(iter(data_loader))
    x = x.to(device)
    y = y.to(device)

    with torch.no_grad():
        logits = model(x)
        preds = logits.argmax(dim=1)

    x = x.cpu()
    y = y.cpu()
    preds = preds.cpu()

    n = min(max_images, x.size(0))
    cols = min(8, n)
    rows = (n + cols - 1) // cols

    plt.figure(figsize=(2.5 * cols, 2.8 * rows))
    for i in range(n):
        img = x[i] * std_t + mean_t
        img = torch.clamp(img, 0.0, 1.0)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img.permute(1, 2, 0).numpy())
        gt = class_names[y[i].item()]
        pd = class_names[preds[i].item()]
        ok = "✓" if gt == pd else "✗"
        plt.title(f"GT: {gt}\nPred: {pd} {ok}", fontsize=8)
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:

set_seed(SEED, deterministic=False)

weak_tf, strong_tf, mean, std = get_transforms()

base_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True
)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_set = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=test_transform
)
test_loader = DataLoader(
    test_set,
    batch_size=256,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

class_names = test_set.classes


CIFAR-10 mean: [0.49140000343322754, 0.4821999967098236, 0.4465000033378601], std: [0.20229999721050262, 0.19939999282360077, 0.20100000500679016]


100%|██████████| 170M/170M [00:15<00:00, 11.3MB/s] 


In [ ]:
def run_experiments(
    label_budgets=LABEL_BUDGETS,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    mu=MU,
):
    results = {}

    for labels_per_class in label_budgets:
        print(f"\n========== {labels_per_class} rótulos por classe ==========")

        labeled_idx, unlabeled_idx = split_labeled_unlabeled(
            base_train, labels_per_class, num_classes=10
        )

        labeled_dataset = LabeledCIFAR10(
            root=DATA_ROOT,
            idxs=labeled_idx,
            transform=weak_tf,
            download=False,
        )
        unlabeled_dataset = UnlabeledCIFAR10(
            root=DATA_ROOT,
            idxs=unlabeled_idx,
            weak_transform=weak_tf,
            strong_transform=strong_tf,
            download=False,
        )

        # Batch sizes: B para supervisionado, μB para não-supervisionado
        labeled_batch_size = min(batch_size, len(labeled_dataset))
        unlabeled_batch_size = min(batch_size * mu, len(unlabeled_dataset))

        print(f"Batch sizes - Supervised: {labeled_batch_size}, Unsupervised: {unlabeled_batch_size}")

        labeled_loader = DataLoader(
            labeled_dataset,
            batch_size=labeled_batch_size,
            shuffle=True,
            num_workers=4,
            drop_last=False,
            pin_memory=True,
        )

        unlabeled_loader = DataLoader(
            unlabeled_dataset,
            batch_size=unlabeled_batch_size,
            shuffle=True,
            num_workers=4,
            drop_last=True,
            pin_memory=True,
        )

        sup_model = build_resnet18_cifar10()
        sup_trainer = SupervisedTrainer(
            model=sup_model,
            labeled_loader=labeled_loader,
            test_loader=test_loader,
            device=DEVICE,
            num_epochs=num_epochs,
        )
        sup_trainer.train(num_epochs=num_epochs, log_every=max(1, num_epochs // 5))
        sup_acc = sup_trainer.history["acc"][-1]
        print(f"[Resumo] Supervisionado ({labels_per_class}/cls): {sup_acc:.2f}%")

        fm_model = build_resnet18_cifar10()



        fm_trainer = FixMatchTrainer(
            model=fm_model,
            labeled_loader=labeled_loader,
            unlabeled_loader=unlabeled_loader,
            test_loader=test_loader,
            device=DEVICE,
            lambda_u=1.0,
            tau=0.95,
            num_epochs=num_epochs,
            mu=mu,
            lr=0.03,
            weight_decay=5e-4,
        )
        fm_trainer.train(num_epochs=num_epochs)
        fm_acc = fm_trainer.evaluate()
        print(f"[Resumo] FixMatch ({labels_per_class}/cls): {fm_acc:.2f}%")

        if ENABLE_VISUALIZATION:
            visualize_labeled_predictions(
                sup_trainer.model,
                DEVICE,
                labeled_loader,
                mean, std,
                class_names=class_names,
                title=f"Supervised - {labels_per_class} lbl/cls"
            )

            visualize_labeled_predictions(
                fm_trainer.model,
                DEVICE,
                labeled_loader,
                mean, std,
                class_names=class_names,
                title=f"FixMatch - {labels_per_class} lbl/cls"
            )

            visualize_pseudo_labels_fixmatch(
                fm_trainer,
                mean, std,
                unlabeled_loader,
                class_names=class_names,
                tau=fm_trainer.tau,
                title=f"FixMatch {labels_per_class} lbl/cls - pseudo-rótulos"
            )

        if ENABLE_PSEUDO_STATS:
            fm_trainer.pseudo_label_stats()

        results[labels_per_class] = {
            "supervised_acc": sup_acc,
            "fixmatch_acc": fm_acc,
            "supervised_history": sup_trainer.history,
            "fixmatch_history": fm_trainer.history,
        }

    return results


In [ ]:
results = run_experiments()

plot_overall_accuracy(results)

for lpc, res in results.items():
    plot_supervised_history(res["supervised_history"], title=f"Sup {lpc}/cls")
    plot_fixmatch_history(res["fixmatch_history"], title=f"FixMatch {lpc}/cls")



========== 25 rótulos por classe ==========
⚠️  Poucos dados rotulados detectados! Ajustando batch sizes para manter proporção 7:1
Batch sizes - Supervised (B): 250, Unsupervised (μB): 1750
Proporção real: 7.0:1 (target: 7:1)
[Supervised] Epoch 1/70 - Loss: 2.4400 - Acc: 10.01%


KeyboardInterrupt: 